# Bayesian Filtering: Prediction Step

## Review of Algorithm Steps

For a single time step, Bayesian filtering involves:

1. Prediction of state distribution (forming prior belief)
2. Receipt of a measurement (a.k.a. evidence)
3. Update Belief (apply Baye's Rule)

## Previously On Bayesian Filtering...

In the last section we covered how we can update our prior belief about our current position based on the evidence provided by the GPS. 

$$
p(x_0 \mid z_0) = \frac{p(x_0)p(z_0 \mid x_0)}{\int{p(x)p(z_0 \mid x)}dx}
$$

Where 
- $p(x_0)$ was our prior belief
- $p(z_0 \mid x_0)$ was the likelihood of the measurement occured given different potential true positions
- $\int{p(x)p(z_0 \mid x)}dx$ was the total probability of the evidence, regardless of true state. It served to normalize the updated belie, creating a valid probability distribution that sums to 1 across its support.


## Prediction: Evolving Belief Forward in Time

The prediction step of the algorithm describes how we use our belief about what's happening now to inform our belief about the future. Remember that belief is described by probability distributions, not discrete values. 

Let's start with a discrete example. We'll assume that we were constrained to 3 potential positions at time $t = 0$

- $X_0 = a$ 
- $X_0 = b$
- $X_0 = c$

These are our previous potential states. Now time has passed and $t = 1$. We are interested in our next position $x_1$. The probability of transition to $x_1$ from each of our three potential positions are:

- $P(x_1 \mid a) = 0.2$
- $P(x_1 \mid b) = 0.3$
- $P(x_1 \mid c) = 0.1$

Then, to get total probability, **which is our prediction of $x_1$** we sum the probabilities of all potential paths to $x_1$ from the previous locations:

<br>
$$P(x_1) = P(a)P(x_1 \mid a) + P(b)P(x_1 \mid b) + P(c)P(x_1 \mid c)$$
<br>

Where each term in the sum says "The probability that we were at some position AND we transition from that position to $x_1$". For example $P(a)P(x_1 \mid a)$ is the probability that we found ourselves at position $a$ AND we transitioned to position $x_1$. Writing this discrete sum in more formal notation, we'd say:

<br>
$$
p(X_1 = x_1) = \sum_{x_0 \in X_0}p(x_1 \mid x_0)p(x_0)
$$
<br>

And if our case was continuous (not limited to a finite number of discrete initial positions), the sum becomes an integral.

<br>
$$
p(X_1 = x_1) = \int p(x_1 \mid x_0)p(x_0)dx_0
$$
<br>

Remember that at $t_0$ we we updated our belief p(X_0) when we received a GPS measurement $z_0$. We applied Bayes Law to obtain $p(X_0 | z_0)$ and therefore our prediction formula becomes 

<br>
$$
p(X_1 = x_1) = \int p(x_1 \mid x_0)p(x_0 \mid z_0)dx_0
$$
<br>

#### And...

$P(X_1)$ becomes our *new prior* which we hope to update when a measurement is received for that timestep. 



## Big Question
At this point, the formula might make sense, but how we use it in practice still seems like a big mystery. Where do the transition probabilites $p(x_k \mid x_{k - 1})$ come from? The answer is very similar to how we figured out our likelihood function in the update step. If you recall, we figured out how to obtain the likelihood by modeling the GPS behavior $$Z_t = x_t + V$$ where $V \sim \mathcal{N}(0, \sigma^2)$ was the random GPS error and $x_t$ was our unknown true state. This is what we called the *measurement model* and it gave us our likelihood function. Likewise, we need a probabalistic *process model* to tell us how we expect our previous position to evolve at each time step. 

### Example: Constant Velocity

Let's say that we assume that we are moving along at a constant velocity of $v$ unit of distance per unit of time, then we might model our movement as

$$
X_k = X_{k - 1} + v + W
$$


where $W \sim \mathcal{N}(0, \sigma^2)$ represents uncertainty in our process model. It accounts for effects that are not captured by our assumption of constant velocity. As we did with the measurement model, we ask "what happens if we choose a candidate true state $X_{k - 1} = x_{t - 1}$? Then, what can we say about

<br>
$$p(X_k | X_{k - 1} = x_{k - 1})$$ 
<br>

Well, since we fixed $X_{k - 1} = x_{t - 1}$, the only source of randomness is the zero mean error term $W$. Therefore, the mean of $X_k$ is simply the the mean of $W$ shifted by $x_{k - 1} + v$ through addition to get

<br>
$$
p(X_k | X_{k - 1} = x_{k - 1}) \sim \mathcal{N}(x_{k - 1} + v, \sigma^2)
$$
<br>

And, like last time, we look up or know the form of a normal distribution is

<br>
$$
p(x_k; \mu, \sigma) = \frac{1}{\sqrt{2 \pi \sigma^2}} \exp \bigg(-\frac{(x_k - \mu)^2}{2 \sigma^2} \bigg)
$$
<br>

And if we substitute $\mu = x_{k - 1} + v$, we get 

$$
p(x_k; \mu, \sigma) = \frac{1}{\sqrt{2 \pi \sigma^2}} \exp \bigg(-\frac{( x_{k} - k_{k - 1} - v)^2}{2 \sigma^2} \bigg)
$$
